# 04_02 Feed-forward network with GELU

*📖 Book: section 4.3*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/gpt.py`
>
> **This notebook:** After attention has mixed in the earlier letters, each letter needs a step of its own to think about the result: a small 2-layer network with a GELU curve.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What the feed-forward step is: a small 2-layer network, applied to each letter on its own
- Why it expands to 4 times the width and shrinks back
- What GELU is, and how it differs from ReLU
- How many weights it has, compared with attention and our whole model

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **feed-forward network** | a small network of layers where numbers only move forward, one layer after the next; in a block it works on each letter separately | 32 → 128 → 32 numbers |
| **activation function** | a small curve applied to every number after a layer, so that stacked layers can do more than one straight layer could | ReLU, GELU |
| **ReLU** | an activation: negative numbers become 0, positive numbers stay | ReLU(-2) = 0, ReLU(3) = 3 |
| **GELU** | an activation like ReLU, but smooth: small negative numbers are shrunk, not cut to 0 | GELU(-1) ≈ -0.16 |

## 3. The idea

### ❓ What's left to do once attention has mixed the letters?

Attention (chapter 3) lets each letter **gather** information from the letters before it. The feed-forward step then lets each letter think about what it gathered. It's applied to every letter on its own, so there's no looking at neighbours here.

```
attention:     ἄ ν δ ρ α   →  every letter mixes in the earlier ones
feed-forward:  ἄ   ν   δ   ρ   α   →  each letter is processed alone, by the same small network
```

It's a small network (see [01_02](../01_foundations/01_02_neural_networks.ipynb)) of two layers with a curve between them. The first layer expands each letter's 32 numbers to 4 times as many, 128, which gives it more room to think. Then the curve is applied. Then the second layer shrinks it back to 32, so the shape still fits the next floor.

```
32 numbers → [layer 1] → 128 numbers → [GELU curve] → 128 numbers → [layer 2] → 32 numbers
```

### ❓ Why is there a curve in the middle?

Two layers one after the other, with nothing between them, are secretly just one layer. Multiplying by one table of numbers and then by another is the same as multiplying once by a combined table. The curve (an **activation function**) breaks that, so the two layers can do something one layer can't. The curve works on every number separately.

The simple, classic curve is ReLU: negative numbers become 0 and positive numbers stay as they are. GPT models use a smoother one, GELU. Let's compute both for a few numbers.

👉 **Predict first:** what will ReLU give for -2, -1, 0, 1, 2? What do you guess GELU gives for -1?

In [ ]:
import torch
from torch import nn

numbers = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("number:", numbers.tolist())
print("ReLU:  ", nn.ReLU()(numbers).tolist())
print("GELU:  ", [round(v, 3) for v in nn.GELU()(numbers).tolist()])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

ReLU gives `0, 0, 0, 1, 2`: every negative number is cut to 0. GELU gives `-0.046, -0.159, 0, 0.841, 1.954`. So the positives are almost unchanged, and the negatives aren't thrown away, only shrunk. (The book writes GELU with a longer formula that's very close. PyTorch's `nn.GELU()` is the exact one, and that's the one we use.)

</details>

### ❓ Can I see the two curves?

The input is on the left-right axis, the output up-down. ReLU has a sharp corner at 0. GELU is smooth, and dips a little below 0 before it rises.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

xs = torch.linspace(-4, 4, 400)
relu_curve, gelu_curve = nn.ReLU()(xs), nn.GELU()(xs)
low = gelu_curve.argmin()  # the lowest point of the dip

fig = studio_figure("Activation functions", "ReLU cuts, GELU curves", (9, 6))
ax = studio_axes(fig)
ax.plot(xs, relu_curve, color=MUTED, linewidth=3, linestyle="--", label="ReLU")
ax.plot(xs, gelu_curve, color=ACCENT, linewidth=3.5, label="GELU")
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.axvline(0, color=MUTED, linewidth=0.8)
ax.annotate(
    f"lowest dip: GELU({xs[low]:.2f}) = {gelu_curve[low]:.2f}",
    (xs[low], gelu_curve[low]), (-3.9, 1.8), color=INK, fontsize=10,
    family="monospace", arrowprops={"arrowstyle": "->", "color": INK},
)
ax.set_xlabel("number in")
ax.set_ylabel("number out")
ax.set_ylim(-1, 4)
ax.legend(frameon=False, labelcolor=INK, loc="lower right")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

For big positive numbers the two lines are the same, and for big negative numbers both are near 0. In between, GELU is smooth where ReLU has a corner, and it dips to about -0.17. Why does smooth matter? A smooth curve gives gradients ([01_03](../01_foundations/01_03_how_a_program_learns.ipynb)) that change gently, and that suits learning.

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: build the feed-forward step

Three parts in a row: a layer from 32 to 128 numbers, the GELU curve, and a layer from 128 back to 32. `nn.Sequential` runs its parts one after another. We feed in the five rows of `ἄνδρα`.

👉 **Predict first:** what shape comes out for 5 letters of 32 numbers each?

In [ ]:
import torch
from torch import nn

from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer(load_books(BOOKS_DIR)[0])
print("vocabulary size:", tokenizer.vocab_size)

In [ ]:
torch.manual_seed(0)
embedding = nn.Embedding(tokenizer.vocab_size, 32)  # a new, untrained table
ids = torch.tensor([tokenizer.encode("ἄνδρα")])  # shape (1, 5): 1 text, 5 letters
x = embedding(ids)  # shape (1, 5, 32): 5 letters, 32 numbers each
print("ids:", tuple(ids.shape), " x:", tuple(x.shape))

In [ ]:
feed_forward = nn.Sequential(
    nn.Linear(32, 4 * 32),  # expand: 32 → 128
    nn.GELU(),
    nn.Linear(4 * 32, 32),  # shrink: 128 → 32
)

inside = feed_forward[0](x)
print("in:        ", tuple(x.shape))
print("expanded:  ", tuple(inside.shape))
print("out:       ", tuple(feed_forward(x).shape))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(1, 5, 32)` in, `(1, 5, 128)` in the middle, `(1, 5, 32)` out. Same shape out as in, and that's what lets blocks stack.

</details>

### ❓ Step 2: is each letter really processed on its own?

Let's test it. Put in only the first letter, `ἄ`, and compare with the first row of the full run. If the letters really are separate, the two must be the same.

👉 **Predict first:** same, or different?

In [ ]:
alone = feed_forward(x[:, :1])  # only the first letter
together = feed_forward(x)[:, :1]  # the first row of the full run
print("same?", torch.allclose(alone, together, atol=1e-6))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`True`. The other four letters had no effect, so the letters don't talk to each other here. (Talking is attention's job.)

</details>

### ❓ Step 3: how many weights?

Each `nn.Linear` has a table of weights and a list of biases (one added number per output):

```
layer 1:  32 × 128 weights + 128 biases = 4,096 + 128 = 4,224
layer 2: 128 × 32  weights +  32 biases = 4,096 +  32 = 4,128
```

👉 **Predict first:** the total. Is it more or less than our whole model's 11,840?

In [ ]:
total = sum(p.numel() for p in feed_forward.parameters())
print("feed-forward weights:", f"{total:,}")
print("attention in model.py: 3,072 (query + key + value)")
print("whole model.py:        11,840")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

8,352 weights. That's more than two and a half times the attention (3,072), and about 70% of the size of our entire model. (The biases are why the total is a little more than 2 × 4,096.)

</details>

### ❓ Your turn: what does a feed-forward step cost for rows of 128 numbers?

Bigger models use longer rows. Say each letter has 128 numbers, four times our 32. The step still expands to 4 times the width and shrinks back. Count its weights with plain arithmetic, biases included.

👉 **Predict first:** four times as wide. Four times the weights?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
width = 128

# 👉 do: the weights of Linear(width, 4 * width) plus those of Linear(4 * width, width)
# 💡 hint: a Linear layer has in × out weights, plus one bias for each output
wide_weights = None

In [ ]:
from homer_ai.lessons import check

wide = nn.Sequential(nn.Linear(128, 512), nn.GELU(), nn.Linear(512, 128))
check(
    "feed-forward weights at width 128",
    wide_weights,
    sum(p.numel() for p in wide.parameters()),
    coach="Got 131,072? Those are the weights without the biases. Each Linear "
    "layer also has one bias per output: 512 for the first, 128 for the second.",
)

<details><summary>💡 Hint</summary>

For our width of 32 the sum is `(32 × 128 + 128) + (128 × 32 + 32) = 8,352`. Do the same with 128 and 512.

</details>

<details><summary>✅ Answer</summary>

```python
width = 128

expand = width * 4 * width + 4 * width
shrink = 4 * width * width + width
wide_weights = expand + shrink
```

131,712, nearly 16 times our 8,352. The row got 4 times as long and the middle layer too, and 4 × 4 = 16. So the cost grows with the **square** of the width, which is why wide models get big so fast.

</details>

### ❓ Does our `model.py` have a feed-forward step?

No. Between attention and `head` it has nothing, and we don't change `model.py` here. The feed-forward step lives in the block: in [04_04](04_04_one_block.ipynb) it becomes half of it.

## 5. Recap

- The feed-forward step is a 2-layer network applied to each letter alone: 32 → 128 → 32, with a GELU curve between.
- GELU is a smooth ReLU: ReLU cuts negatives to 0, GELU shrinks them (GELU(-1) ≈ -0.16).
- It has 8,352 weights, more than twice the attention in our model. Our `model.py` has no feed-forward step.

**You can now:**

- build the feed-forward step with `nn.Sequential`: expand, GELU, shrink
- count the weights of a `Linear` layer: in × out, plus one bias per output
- say how GELU differs from ReLU for negative numbers

**Next:** [04_03: Shortcuts](04_03_shortcuts.ipynb): how learning reaches the lower floors.